# 10c - Optimisation de la régression : TousAge_allergie_urgences

Bases :
- cible : le taux de la semaine suivante (S+1), par département
- train 2021-2022, validation 2023, test 2024


Remarques Romain :
- est-ce que la standardisation a créé du bruit ?
- trop de données : est-ce qu'il faut moins de variables ?
- optimisation des hyperparamètres

Plan :
Mettre à l'échelle : rien, standardisation, normalisation
Sélection de variables : SelectKBest, RFE, sélection backward
GridSearchCV sur Ridge, Lasso, ElasticNet


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression, RFE, SequentialFeatureSelector
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Données 

In [3]:
df = pd.read_csv("../data/final_metropole/Merge_final_2.csv", dtype={"code_dept": str})

cible = "TousAge_allergie_urgences"

df["annee"] = df["date_semaine"].str[:4].astype(int)
df["semaine"] = df["date_semaine"].str[-2:].astype(int)
df["sos_present"] = df["sos_present"].astype(int)

df = df.sort_values(["code_dept", "annee", "semaine"]).reset_index(drop=True)

In [ ]:
g = df.groupby("code_dept")[cible]

# taux de la semaine suivante 
df["target"] = g.shift(-1)

# moyenne2
df["moy2_dept"] = g.transform(lambda s: s.rolling(2).mean())

# avec un shift de 51 semaines
df["an_dernier"] = g.shift(51)

# transformation des semaines : sinus et cosinus
df["sem_sin"] = np.sin(2 * np.pi * df["semaine"] / 52)
df["sem_cos"] = np.cos(2 * np.pi * df["semaine"] / 52)

In [5]:
cols = (["moy2_dept", "sem_sin", "sem_cos"]
        # météo
        + ["temperature_kelvin", "precipitation_3h", "humidite_relative",
           "vent_vitesse", "pression_mer", "visibilite"]
        # pollution
        + ["As in PM10", "BaP in PM10", "Cd in PM10", "Ni in PM10", "Pb in PM10",
           "NO2", "O3", "PM10", "PM2.5", "SO2",
           "Chl in PM1", "NO3 in PM1", "NH4 in PM1", "Org in PM1", "SO4 in PM1"]
        # socio (sans les artisans)
        + ["pct_urbain", "pct_cadres", "pct_prof_inter", "pct_employes", "pct_ouvriers",
           "pct_retraites", "pct_sans_activite", "pct_agriculteurs",
           "densite_generalistes", "densite_pediatres", "densite_pneumologues",
           "densite_allergologues", "densite_kines", "sos_present"]
        # prévalence
        + ["prev_Maladie_Respi_Chronique_tsage_9",
           "prev_calc_Maladie_Respi_Chronique_0-14_9",
           "prev_calc_Maladie_Respi_Chronique_15-64_9",
           "prev_calc_Maladie_Respi_Chronique_65+_9"]
        + ["an_dernier"])
print(len(cols), "variables")

43 variables


In [ ]:
# nettoyage
d = df.dropna(subset=cols + ["target"])

# train/ validation/ test
train = d[d["annee"] <= 2022].sort_values(["annee", "semaine"])
val = d[d["annee"] == 2023]
test = d[d["annee"] == 2024]

X_train = train[cols]
y_train = train["target"]
X_val = val[cols]
y_val = val["target"]
print(len(train), len(val), len(test))

5088 4992 4896


In [8]:
# calcul de la MAE, RMSE et du R²
def scores(y_vrai, y_pred):
    return {"MAE": mean_absolute_error(y_vrai, y_pred),
            "RMSE": mean_squared_error(y_vrai, y_pred) ** 0.5,
            "R2": r2_score(y_vrai, y_pred)}

## 2. Régression linéaire : point de départ (la référence)

In [14]:
lr = LinearRegression().fit(X_train, y_train)
depart = scores(y_val, lr.predict(X_val))

print("Train :", scores(y_train, lr.predict(X_train)))
print("Val   :", depart)
print("Test  :", scores(test["target"], lr.predict(test[cols])))

Train : {'MAE': 172.00446298386518, 'RMSE': 236.8513227610065, 'R2': 0.46468303798473387}
Val   : {'MAE': 180.89706105697667, 'RMSE': 244.76887688266763, 'R2': 0.4274005790341473}
Test  : {'MAE': 168.39314464483564, 'RMSE': 229.25871994277162, 'R2': 0.3914198858915555}


Ces métriques serviront de référence pour évaluer l'efficacité de nos optimisations.
Score de référence : R² 0,427

## 3. Mise à l'échelle 

Afin de vérifier que la standardisation et la normalisation ne créent pas de bruit qui pourrait interférer avec notre modèle, notre régression linéaire va être comparée sur des données sans mise à l'échelle, standardisées et normalisées.

In [9]:
resultats = {}

for nom in ["aucune", "standardisation", "normalisation"]:
    if nom == "aucune":
        X_tr = X_train
        X_va = X_val
    elif nom == "standardisation":
        scaler = StandardScaler().fit(X_train)
        X_tr = scaler.transform(X_train)
        X_va = scaler.transform(X_val)
    else:
        scaler = MinMaxScaler().fit(X_train)
        X_tr = scaler.transform(X_train)
        X_va = scaler.transform(X_val)

    lr = LinearRegression().fit(X_tr, y_train)
    ridge = Ridge(alpha=10).fit(X_tr, y_train)
    resultats[nom] = {"Régression linéaire": r2_score(y_val, lr.predict(X_va))}
                     

pd.DataFrame(resultats).round(4)

,aucune,standardisation,normalisation
Régression linéaire,0.4274,0.4274,0.4274


On obtient le même R² dans les trois cas (0,4274). La standardisation faite dans la partie précédente n'a pas crée de bruit.


## 4. Sélection de variables : est-ce que notre modèle est limité par trop de variables ?

Ces trois méthodes permettent d'étudier le rôle de chaque variable dans notre modèle :
- SelectKBest avec f_regression (teste le lien de chaque variable avec notre cible)
- RFE (retire la variable la moins importante du modèle, puis recommence)
- sélection backward (retire une à une les variables qui ne changent rien au modèle)

Le Lasso et l'ElasticNet retirent des variables en mettant leur coefficient à 0. Comparer ces différentes méthodes permet de repérer les variables qui ressortent le plus souvent comme non indispensables à notre modèle.

In [ ]:
# standardisation pour RFE et la sélection backward car ils se basent sur les coefficients
scaler = StandardScaler().fit(X_train)
X_train_sc = scaler.transform(X_train)
X_val_sc = scaler.transform(X_val)

### a. SelectKBest

In [11]:
kbest = {}

for k in [3, 5, 10, 15, 20, 25, 30, 35, 43]:
    sk = SelectKBest(f_regression, k=k).fit(X_train, y_train)
    lr = LinearRegression().fit(sk.transform(X_train), y_train)
    kbest[k] = r2_score(y_val, lr.predict(sk.transform(X_val)))

pd.Series(kbest).round(4)

3     0.3990
5     0.4191
10    0.4274
15    0.4309
20    0.4217
25    0.4224
30    0.4162
35    0.4155
43    0.4274
dtype: float64

In [12]:
# SelectKBest : variables retirées (k = 15)
sk = SelectKBest(f_regression, k=15).fit(X_train, y_train)
gardees_kbest = sk.get_feature_names_out()

retirees_kbest = []
for c in cols:
    if c not in gardees_kbest:
        retirees_kbest.append(c)
retirees_kbest

['sem_sin',
 'precipitation_3h',
 'vent_vitesse',
 'pression_mer',
 'visibilite',
 'As in PM10',
 'Cd in PM10',
 'Ni in PM10',
 'Pb in PM10',
 'PM10',
 'SO2',
 'NH4 in PM1',
 'Org in PM1',
 'pct_cadres',
 'pct_prof_inter',
 'pct_employes',
 'pct_ouvriers',
 'pct_retraites',
 'pct_sans_activite',
 'pct_agriculteurs',
 'densite_pediatres',
 'densite_pneumologues',
 'densite_allergologues',
 'sos_present',
 'prev_Maladie_Respi_Chronique_tsage_9',
 'prev_calc_Maladie_Respi_Chronique_0-14_9',
 'prev_calc_Maladie_Respi_Chronique_15-64_9',
 'prev_calc_Maladie_Respi_Chronique_65+_9']

Les 15 variables les plus liées à notre cible suffisent. C'est le k qui obtient le R² le plus élevé (0,431). Les scores restent très proches.

### b. RFE

In [11]:
rfe_res = {}

for k in [5, 10, 15, 20, 25, 30, 35]:
    rfe = RFE(LinearRegression(), n_features_to_select=k).fit(X_train_sc, y_train)
    lr = LinearRegression().fit(X_train_sc[:, rfe.support_], y_train)
    rfe_res[k] = r2_score(y_val, lr.predict(X_val_sc[:, rfe.support_]))

pd.Series(rfe_res).round(4)

5     0.4220
10    0.4345
15    0.4276
20    0.4246
25    0.4252
30    0.4258
35    0.4280
dtype: float64

In [13]:
# RFE : variables retirées (k = 10)
rfe = RFE(LinearRegression(), n_features_to_select=10).fit(X_train_sc, y_train)
gardees_rfe = X_train.columns[rfe.support_]

retirees_rfe = []
for c in cols:
    if c not in gardees_rfe:
        retirees_rfe.append(c)
retirees_rfe

['sem_sin',
 'temperature_kelvin',
 'precipitation_3h',
 'humidite_relative',
 'vent_vitesse',
 'pression_mer',
 'visibilite',
 'As in PM10',
 'BaP in PM10',
 'Cd in PM10',
 'Ni in PM10',
 'Pb in PM10',
 'NO2',
 'O3',
 'PM10',
 'PM2.5',
 'SO2',
 'Chl in PM1',
 'NO3 in PM1',
 'NH4 in PM1',
 'SO4 in PM1',
 'pct_prof_inter',
 'pct_ouvriers',
 'pct_retraites',
 'pct_sans_activite',
 'pct_agriculteurs',
 'densite_pediatres',
 'densite_pneumologues',
 'densite_allergologues',
 'densite_kines',
 'sos_present',
 'prev_calc_Maladie_Respi_Chronique_15-64_9',
 'prev_calc_Maladie_Respi_Chronique_65+_9']

Pour RFE, les 10 variables le splus liées à notre cible permettent d'obtenir le meilleur score (0,4345). Proche des autre scores. 

### c. Sélection backward


In [12]:
sbs = SequentialFeatureSelector(LinearRegression(), n_features_to_select="auto",
                                direction="backward", cv=TimeSeriesSplit(5), scoring="r2")
sbs.fit(X_train_sc, y_train)

garde = []
for i in range(len(cols)):
    if sbs.support_[i]:
        garde.append(cols[i])

retire = []
for c in cols:
    if c not in garde:
        retire.append(c)

print(len(garde), "variables gardées")
lr = LinearRegression().fit(train[garde], y_train)
backward = scores(y_val, lr.predict(val[garde]))
backward

22 variables gardées


{'MAE': 177.59913356554839,
 'RMSE': 242.12532329104837,
 'R2': 0.43970216864883116}

In [13]:
# variables retirées
retire

['sem_sin',
 'precipitation_3h',
 'humidite_relative',
 'pression_mer',
 'visibilite',
 'As in PM10',
 'Cd in PM10',
 'Ni in PM10',
 'Pb in PM10',
 'NO2',
 'O3',
 'PM2.5',
 'SO2',
 'Chl in PM1',
 'SO4 in PM1',
 'pct_agriculteurs',
 'densite_pneumologues',
 'densite_allergologues',
 'densite_kines',
 'prev_calc_Maladie_Respi_Chronique_15-64_9',
 'prev_calc_Maladie_Respi_Chronique_65+_9']

Backward : garde 22 variables, R² 0,440, donne le meilleur score. 
Les trois méthodes retirent toutes 14 variables, surtout de la pollution (métaux, SO2), sem_sin (axe printemps/automne), attention semaine en sinus est indispensable pour les saisons et de la météo (pluie, pression, visibilité). Elles gardent toutes moy2_dept, sem_cos, an_dernier, pct_urbain et densite_generalistes : ce sont les variables les plus importantes pour notre modèle.